# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.

```
prompt | llm | parser
```

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. 기본 체인: prompt | llm | parser

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 DB"})
print(result)


벡터 DB는 고차원 벡터 데이터를 저장하고 검색하는 데 최적화된 데이터베이스로, 주로 머신러닝 및 인공지능 응용에서 유사성 검색에 사용됩니다.


## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

임베딩은 고차원 데이터를 저차원 공간에 매핑하여 의미 있는 벡터 표현으로 변환하는 기법입니다.

임베딩은 고차원 데이터를 저차원 공간에 매핑하여 의미 있는 벡터 표현을 생성하는 기법으로, 주로 자연어 처리와 추천 시스템에서 사용됩니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기


{'topic': 'RAG'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [ ]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
# 입력값에 "word" 키가 있다고 가정할 때, "upper" 키에 대문자 변환값을 추가하는 체인

from langchain_core.runnables import RunnablePassthrough




{'word': 'hello', 'upper': 'HELLO'}


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [ ]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수



[결과] LangGraph는 자연어 처리(NLP)와 관련된 기술이나 도구로, 언어와 그래프 이론을 결합하여 언어 데이터를 분석하고 처리하는 방법을 제공하는 시스템이나 프레임워크일 수 있습니다. 일반적으로 이러한 시스템은 언어의 구조적 특성을 그래프 형태로 모델링하여, 단어, 문장, 또는 문서 간의 관계를 시각화하고 분석하는 데 사용됩니다.

LangGraph의 구체적인 기능이나 목적은 사용되는 맥락에 따라 다를 수 있으며, 예를 들어, 언어 모델링, 정보 검색, 텍스트 요약, 또는 의미적 관계 분석 등에 활용될 수 있습니다. 

더 구체적인 정보가 필요하시다면, LangGraph의 특정 구현이나 사용 사례에 대해 말씀해 주시면 추가적인 설명을 드릴 수 있습니다.


## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

In [ ]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])






[한국어] RAG는 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 생성 모델을 결합한 접근 방식을 의미합니다. 이 방법은 주로 자연어 처리(NLP) 분야에서 사용되며, 특히 질문 응답 시스템이나 대화형 AI 모델에서 효과적입니다.

RAG의 기본 원리는 다음과 같습니다:

1. **정보 검색**: 사용자가 질문을 입력하면, 시스템은 관련된 정보를 데이터베이스나 문서에서 검색합니다. 이 단계에서는 검색 엔진이나 데이터베이스 쿼리를 사용하여 질문과 관련된 문서나 정보를 찾아냅니다.

2. **정보 생성**: 검색된 정보를 바탕으로, 생성 모델이 자연어로 응답을 생성합니다. 이 과정에서는 검색된 내용을 요약하거나, 질문에 대한 구체적인 답변을 작성합니다.

이러한 방식의 장점은 모델이 사전에 학습한 지식뿐만 아니라, 최신 정보나 특정 도메인에 대한 세부 정보를 활용할 수 있다는 점입니다. 따라서 RAG는 더 정확하고 유용한 응답을 제공할 수 있습니다. 

RAG는 특히 대규모 데이터셋을 활용할 수 있는 환경에서 강력한 성능을 발휘하며, 정보의 정확성과 관련성을 높이는 데 기여합니다.
[English] RAG stands for "Red, Amber, Green," which is a color-coding system used to indicate the status or performance of a project, task, or situation. Each color represents a different level of urgency or concern:

- **Red**: This indicates a serious issue or problem that requires immediate attention. It suggests that the project is off track, facing significant challenges, or at risk of failing to meet its objectives.

## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요.   결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. 출력 파서 생성


# 4. LLM 응답을 받아 단어 수를 세는 함수



# 5. RunnableLambda로 함수 감싸기


# 6. LCEL 체인 구성


# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})



178
--------------------
LangChain은 자연어 처리(NLP)와 관련된 다양한 작업을 수행할 수 있도록 도와주는 프레임워크입니다. 그 중에서도 "Runnable"은 LangChain에서 특정 작업을 수행하는 단위를 의미합니다. 초보자도 이해할 수 있도록 간단하게 설명해볼게요.

### Runnable이란?

1. **작업 단위**: Runnable은 특정한 작업을 수행하는 코드 블록입니다. 예를 들어, 텍스트를 분석하거나, 질문에 대한 답변을 생성하는 등의 작업을 할 수 있습니다.

2. **입력과 출력**: Runnable은 입력을 받아서 처리한 후, 결과를 출력합니다. 예를 들어, 사용자가 "안녕하세요"라는 문장을 입력하면, Runnable은 이 문장을 처리하여 "안녕하세요! 어떻게 도와드릴까요?"와 같은 응답을 생성할 수 있습니다.

3. **조합 가능성**: 여러 개의 Runnable을 조합하여 더 복잡한 작업을 수행할 수 있습니다. 예를 들어, 첫 번째 Runnable이 텍스트를 정리하고, 두 번째 Runnable이 그 텍스트를 분석하는 식으로 연결할 수 있습니다.

### 예시

- **텍스트 변환**: 사용자가 입력한 문장을 대문자로 변환하는 Runnable을 만들 수 있습니다.
- **질문 응답**: 사용자가 질문을 입력하면, 그에 대한 답변을 생성하는 Runnable을 만들 수 있습니다.

### 요약

LangChain의 Runnable은 특정 작업을 수행하는 코드 블록으로, 입력을 받아서 처리한 후 결과를 출력합니다. 여러 Runnable을 조합하여 복잡한 작업을 수행할 수 있는 유용한 도구입니다. 이를 통해 자연어 처리 작업을 더 쉽게 구현할 수 있습니다. 

이해가 되셨나요? 추가적인 질문이 있다면 언제든지 물어보세요!
